# 19 · Policy gradients: REINFORCE, baselines, returns, GAE

**Needed for:** REINFORCE and GAE (RL track, step R1), and everything after: PPO, RLHF, GRPO all estimate the
same policy gradient with different advantage estimates.

$$\nabla_\theta J = \mathbb{E}\Big[\sum_t \nabla_\theta \log \pi_\theta(a_t\mid s_t)\, A_t\Big]$$

Increase the log-probability of actions that turned out better than expected ($A_t > 0$), decrease it for worse.
The whole field is about estimating $A_t$ with low variance and little bias:

| Estimate of $A_t$ | Bias | Variance |
|---|---|---|
| Return $G_t$ (REINFORCE) | none | high |
| $G_t - b$ (baseline) | none | lower |
| TD error $r_t + \gamma V(s_{t+1}) - V(s_t)$ | from $V$ | low |
| GAE($\lambda$): a mix of all of the above | tunable | tunable |

The environment is a tiny corridor written inline, so no `gym` install is needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Discounted returns with episode boundaries

$G_t = r_t + \gamma\, G_{t+1}\,(1 - \text{done}_t)$, computed backwards. The `(1 - done)` stops returns from
leaking across the end of one episode into the next. This is the most common bug in hand-written RL code.

In [ ]:
def discounted_returns(rewards, dones, gamma):
    """rewards, dones: (..., T). Returns G with the same shape."""
    G = torch.zeros_like(rewards)
    running = torch.zeros_like(rewards[..., 0])
    for t in reversed(range(rewards.shape[-1])):
        running = rewards[..., t] + gamma * running * (1 - dones[..., t])
        G[..., t] = running
    return G


def test_returns(fn):
    r = torch.tensor([1.0, 1.0, 1.0, 1.0])
    d = torch.tensor([0.0, 1.0, 0.0, 0.0])                   # episode ends after step 1
    torch.testing.assert_close(fn(r, d, 0.5), torch.tensor([1.5, 1.0, 1.5, 1.0]))
    torch.testing.assert_close(fn(r[None].expand(3, 4), d[None].expand(3, 4), 0.5)[2], torch.tensor([1.5, 1.0, 1.5, 1.0]))


test_returns(discounted_returns)
print("returns ok")

## 2. GAE

$$\delta_t = r_t + \gamma V(s_{t+1})(1-\text{done}_t) - V(s_t), \qquad A_t = \delta_t + \gamma\lambda(1-\text{done}_t)A_{t+1}$$

$\lambda = 0$ gives the one-step TD error (low variance, biased by $V$); $\lambda = 1$ gives $G_t - V(s_t)$
(unbiased, high variance). PPO typically uses $\lambda = 0.95$. The value target for training $V$ is $A_t + V(s_t)$.

`values` has one extra entry at the end: the value of the state after the last step (0 if it's terminal).

In [ ]:
def gae(rewards, values, dones, gamma, lam):
    """rewards, dones: (..., T); values: (..., T+1). Returns advantages (..., T)."""
    T = rewards.shape[-1]
    adv = torch.zeros_like(rewards)
    running = torch.zeros_like(rewards[..., 0])
    for t in reversed(range(T)):
        not_done = 1 - dones[..., t]
        delta = rewards[..., t] + gamma * values[..., t + 1] * not_done - values[..., t]
        running = delta + gamma * lam * not_done * running
        adv[..., t] = running
    return adv


def test_gae(fn):
    torch.manual_seed(0)
    r = torch.randn(6)
    v = torch.randn(7)
    d = torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0, 0.0])
    v_boot = v.clone()
    v_boot[-1] = 0.0                                          # no bootstrap after the final step
    delta = r + 0.9 * v_boot[1:] * (1 - d) - v_boot[:-1]
    torch.testing.assert_close(fn(r, v_boot, d, 0.9, 0.0), delta)                                   # lambda = 0: TD error
    torch.testing.assert_close(fn(r, v_boot, d, 0.9, 1.0), discounted_returns(r, d, 0.9) - v_boot[:-1])  # lambda = 1


test_gae(gae)
print("GAE reduces to TD error at lambda=0 and to G - V at lambda=1")

## 3. A corridor environment, vectorized

Positions 0 … 9. Start at 0; actions left/right; reaching 9 gives +1 and ends the episode; every step costs 0.01;
at most 30 steps. A random policy rarely makes it. Many episodes run in parallel; `alive` marks steps that belong to an unfinished episode, and all
losses are averaged over alive steps only.

In [ ]:
N_POS, MAX_STEPS = 10, 30


def rollout(policy, B):
    s = torch.zeros(B, dtype=torch.long)
    alive = torch.ones(B, dtype=torch.bool)
    buf = {k: [] for k in ("obs", "act", "logp", "rew", "done", "alive")}
    for _ in range(MAX_STEPS):
        obs = F.one_hot(s, N_POS).float()
        dist = torch.distributions.Categorical(logits=policy(obs))
        a = dist.sample()
        nxt = (s + 2 * a - 1).clamp(0, N_POS - 1)
        reached = nxt == N_POS - 1
        rew = torch.where(reached, 1.0, -0.01) * alive
        buf["obs"].append(obs)
        buf["act"].append(a)
        buf["logp"].append(dist.log_prob(a))
        buf["rew"].append(rew)
        buf["done"].append((reached & alive).float())
        buf["alive"].append(alive.float())
        s = torch.where(alive, nxt, s)
        alive = alive & ~reached
    return {k: torch.stack(v, dim=1) for k, v in buf.items()}     # each (B, T, ...)


policy = nn.Sequential(nn.Linear(N_POS, 32), nn.Tanh(), nn.Linear(32, 2))
batch = rollout(policy, 4)
print("returns of 8 random episodes:", [round(x, 2) for x in rollout(policy, 8)["rew"].sum(1).tolist()])

## 4. What a baseline buys: lower gradient variance

Subtracting a baseline doesn't change the *expected* gradient, only its noise. It matters most when returns share
a large common offset. Here returns already straddle zero (-0.3 for failure, about 0.8 for success), so add a
constant `offset` to every return: say, +5 just for finishing the episode. That doesn't change which actions are
better, but without a baseline every sampled action gets pushed up, and only the noise tells them apart.

First measure gradient variance at a fixed policy over 50 batches, then count training iterations to reach an
average return of 0.8 (best possible: 1 - 8 × 0.01 = 0.92).

In [ ]:
def reinforce_loss(pol, B, gamma, use_baseline, offset=0.0):
    b = rollout(pol, B)
    G = discounted_returns(b["rew"], b["done"], gamma) + offset
    if use_baseline:
        baseline = (G * b["alive"]).sum(0) / b["alive"].sum(0).clamp(min=1)   # mean return at each time step
        G = G - baseline
    return -(b["logp"] * G * b["alive"]).sum() / b["alive"].sum(), b


torch.manual_seed(0)
pol = nn.Sequential(nn.Linear(N_POS, 32), nn.Tanh(), nn.Linear(32, 2))
for offset in (0.0, 5.0):
    for use_baseline in (False, True):
        grads = []
        for _ in range(50):
            pol.zero_grad()
            reinforce_loss(pol, 32, 0.99, use_baseline, offset)[0].backward()
            grads.append(torch.cat([p.grad.flatten() for p in pol.parameters()]))
        print(f"offset {offset:.0f}, baseline={str(use_baseline):5s}: gradient variance {torch.stack(grads).var(0).sum().item():.5f}")


def iters_to_solve(use_baseline, seed, offset, target=0.8, iters=300, B=32):
    torch.manual_seed(seed)
    pol = nn.Sequential(nn.Linear(N_POS, 32), nn.Tanh(), nn.Linear(32, 2))
    opt = torch.optim.Adam(pol.parameters(), lr=1e-2)
    recent = []
    for it in range(iters):
        loss, b = reinforce_loss(pol, B, 0.99, use_baseline, offset)
        opt.zero_grad()
        loss.backward()
        opt.step()
        recent = (recent + [b["rew"].sum(1).mean().item()])[-10:]
        if len(recent) == 10 and sum(recent) / 10 >= target:
            return it
    return "never"


for use_baseline in (False, True):
    print(f"offset 5, baseline={str(use_baseline):5s}: iterations to solve, 5 seeds:",
          [iters_to_solve(use_baseline, s, offset=5.0) for s in range(5)])

## Things to be able to explain

- Why doesn't subtracting a baseline bias the gradient? ($\mathbb{E}[\nabla \log\pi(a\mid s)\, b(s)] = 0$ because
  the probabilities sum to one.)
- Truncation vs termination: if an episode is cut off at `MAX_STEPS`, the true return continues. Proper code
  bootstraps with $V(s_T)$ there instead of treating it as terminal. (This notebook treats it as terminal for
  simplicity.)
- In LLM RL, what is a "step" and what is an "episode"? (Usually one token is an action, one response is an
  episode, and the reward arrives only at the end.)

## Drill

In [ ]:
def my_discounted_returns(rewards, dones, gamma):
    raise NotImplementedError


def my_gae(rewards, values, dones, gamma, lam):
    raise NotImplementedError


drill(test_returns, my_discounted_returns)
drill(test_gae, my_gae)